# Process Intelligence — walkthrough **executando** o pipeline

Este notebook corre **o mesmo processo** que a CLI do projeto:

- `python scripts/run_pipeline.py`
- ou `sian-pipeline` (após `pip install -e .`)

Ou seja: **`app.pipeline.runner.run_pipeline`** — descoberta SX3 → enriquecimento → event log candidato → fluxo → validação → escrita em disco e DuckDB.

As secções seguintes **inspecionam** os artefatos **recém-gerados** (não são uma segunda implementação da lógica).

**Pré-requisitos:** `pip install -e .` na raiz do repositório; ficheiro CSV SX3; DuckDB local com `stg_*` (laboratório).


## Visão do pipeline (mesmo fluxo do código)

```mermaid
flowchart LR
  D[DISCOVERY] --> M[MODEL]
  M --> P[PROCESS]
  P --> V[VALIDATION]
  V --> R[PRESENTATION]
```

Tudo isto é executado **dentro** de `run_pipeline` (`src/app/pipeline/runner.py`).


In [ ]:
# --- Configuração (ajuste antes de correr) ---
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from app.lab import config as lab_config
from app.paths import repo_root
from app.pipeline.runner import run_pipeline
from app.presentation.export_diagram import (
    load_event_log_json,
    load_process_flow_json,
    load_sx3_candidates_json,
)
from app.process.ordering import build_edges, sort_events_for_flow
from app.validation.validate_sequences import validate_linear_edges

BASE_DIR = repo_root()

# Entradas: mesmo padrão do runner (ver parse_args em app.pipeline.runner)
SX3_CSV = BASE_DIR / "data" / "others" / "SX3010_202603231122.csv"
DUCKDB_PATH = lab_config.DUCKDB_PATH
OUTPUT_DIR = BASE_DIR / "data" / "outputs" / "sx3_semantic"

# Se True, executa o pipeline completo (pode demorar; grava DuckDB + JSON).
# Se False, apenas lê artefatos já existentes em OUTPUT_DIR (útil para rever análises).
RUN_PIPELINE = True

DOMAIN = "cp"
MIN_SCORE = 0.0

display(Markdown("### Caminhos"))
display(Markdown(f"- **BASE_DIR:** `{BASE_DIR}`"))
display(Markdown(f"- **SX3_CSV:** `{SX3_CSV}`"))
display(Markdown(f"- **DUCKDB_PATH:** `{DUCKDB_PATH}`"))
display(Markdown(f"- **OUTPUT_DIR:** `{OUTPUT_DIR}`"))
display(Markdown(f"- **RUN_PIPELINE:** `{RUN_PIPELINE}`"))


In [ ]:
# Pré-requisitos
missing = []
if not SX3_CSV.exists():
    missing.append(str(SX3_CSV))
if not DUCKDB_PATH.exists():
    missing.append(str(DUCKDB_PATH))

if missing:
    raise FileNotFoundError(
        "Ficheiros em falta: "
        + ", ".join(missing)
        + "\nPara DuckDB: rode o laboratório (ex.: ./scripts/run_local_lab.sh) ou coloque o .duckdb esperado."
    )
display(Markdown("✓ SX3 e DuckDB encontrados."))


## Execução — `app.pipeline.runner.run_pipeline`

A lista `PIPELINE_ARGS` reproduz o mesmo tipo de chamada que `scripts/run_pipeline.py` faz por defeito (`--build-event-log --build-process-flow --validate`), com caminhos explícitos.

Equivalente na shell:

```bash
python scripts/run_pipeline.py \
  --sx3 <SX3_CSV> \
  --duckdb-path <DUCKDB> \
  -o <OUTPUT_DIR> \
  --domain cp \
  --min-score 0.0 \
  --build-event-log \
  --build-process-flow \
  --validate
```

(`--top-attributes-per-event` tem default 8 no runner; não é obrigatório passar.)


In [ ]:
if RUN_PIPELINE:
    PIPELINE_ARGS = [
        "--sx3",
        str(SX3_CSV),
        "--duckdb-path",
        str(DUCKDB_PATH),
        "-o",
        str(OUTPUT_DIR),
        "--domain",
        DOMAIN,
        "--min-score",
        str(MIN_SCORE),
        "--build-event-log",
        "--build-process-flow",
        "--validate",
    ]
    exit_code = run_pipeline(PIPELINE_ARGS)
    display(Markdown(f"**Pipeline concluído.** Código de saída: `{exit_code}` (0 = sucesso)."))
    display(Markdown(f"Artefatos em `{OUTPUT_DIR}` e tabelas no DuckDB (`sx3_event_candidates`, `event_log_candidates`, `process_flow_snapshot`, …)."))
else:
    display(Markdown("**RUN_PIPELINE = False** — a usar apenas ficheiros já existentes em OUTPUT_DIR."))


## Inspeção DISCOVERY — candidatos (saída do mesmo run)

Os ficheiros abaixo foram **escritos pelo pipeline** acima (ou numa execução anterior no mesmo diretório).


In [ ]:
PATH_SX3 = OUTPUT_DIR / "sx3_event_candidates.json"
if not PATH_SX3.exists():
    raise FileNotFoundError(f"Gere o pipeline antes ou verifique OUTPUT_DIR: {PATH_SX3}")

sx3_rows = load_sx3_candidates_json(PATH_SX3)
df_sx3 = pd.DataFrame(sx3_rows)
cols_show = [
    "activity",
    "source_table",
    "source_column",
    "heuristic_score",
    "column_role",
    "dbt_classification",
]
cols_show = [c for c in cols_show if c in df_sx3.columns]
display(df_sx3[cols_show].head(15))

if "heuristic_score" in df_sx3.columns:
    display(Markdown("### Top por heuristic_score"))
    display(df_sx3.nlargest(12, "heuristic_score")[cols_show])
if "activity" in df_sx3.columns:
    display(df_sx3.groupby("activity", dropna=False).size().sort_values(ascending=False).to_frame("n_candidatos"))


## Inspeção MODEL — event log candidato (`build_event_log` no runner)


In [ ]:
PATH_EVENT_LOG = OUTPUT_DIR / "event_log_candidates.json"
ev_rows = load_event_log_json(PATH_EVENT_LOG)
df_ev = pd.DataFrame(ev_rows)

def _et_row(et):
    if et is None or (isinstance(et, float) and pd.isna(et)):
        return None
    if isinstance(et, dict):
        return f"{et.get('table')}.{et.get('column')}"
    return str(et)

if "event_time" in df_ev.columns:
    df_ev = df_ev.copy()
    df_ev["event_time_ref"] = df_ev["event_time"].map(_et_row)

show = [c for c in ["activity", "event_status", "event_time_ref", "confidence", "dbt_alignment"] if c in df_ev.columns]
display(df_ev[show])


## Inspeção PROCESS + PRESENTATION — `process_flow.json` + Mermaid

Gerado pelo mesmo `run_pipeline` (`build_process_flow` + `write_process_flow_artifacts`).


In [ ]:
PATH_FLOW = OUTPUT_DIR / "process_flow.json"
flow = load_process_flow_json(PATH_FLOW)
nodes = flow.get("nodes") or []
edges = flow.get("edges") or []
unrel = flow.get("unreliable_events") or []

df_nodes = pd.DataFrame(nodes)
if not df_nodes.empty:
    display(df_nodes[[c for c in ["activity", "event_status", "confidence", "dbt_alignment"] if c in df_nodes.columns]])
display(Markdown("### Arestas"))
display(pd.DataFrame(edges))
if unrel:
    display(pd.DataFrame(unrel))
mermaid = flow.get("mermaid") or ""
if mermaid:
    display(Markdown(f"```mermaid\n{mermaid}\n```"))

seq_issues = validate_linear_edges(edges)
display(Markdown("### `validate_linear_edges` (app.validation)"))
display(seq_issues if seq_issues else "OK: cadeia linear.")


## Relatório de validação (escrito pelo runner com `--validate`)

Ficheiro: `validation/validation_report.json` dentro de OUTPUT_DIR.


In [ ]:
vpath = OUTPUT_DIR / "validation" / "validation_report.json"
if vpath.exists():
    with vpath.open(encoding="utf-8") as f:
        rep = json.load(f)
    display(rep)
else:
    display(Markdown("Sem relatório (rode o pipeline com `--validate`)."))


## Simulação em memória — filtrar eventos e reordenar

Usa `app.process.ordering` sobre o **mesmo** `event_log_candidates` carregado (dados do pipeline).


In [ ]:
def filter_events(ev_list, drop_confidence=("BAIXA",)):
    out = []
    for e in ev_list:
        if e.get("event_status") != "OK":
            continue
        if not isinstance(e.get("event_time"), dict):
            continue
        if str(e.get("confidence") or "") in drop_confidence:
            continue
        out.append(e)
    return out

filtered = filter_events(ev_rows)
ordered = sort_events_for_flow(filtered)
edges_sim = build_edges(ordered)
display(Markdown(f"**Filtrado:** {len(ordered)} nós, {len(edges_sim)} arestas vs {len(edges)} no ficheiro"))
display(pd.DataFrame(edges_sim))


## Resumo

| O que correu | Onde está no código |
|--------------|---------------------|
| Pipeline completo | `app.pipeline.runner.run_pipeline` |
| Mesma CLI | `scripts/run_pipeline.py`, `sian-pipeline` |

O notebook **não duplica** regras de negócio: chama o runner e depois analisa outputs.
